# Fase 1 — Planejamento do objetivo e do modelo

**Objetivo da fase:** definir a tarefa do modelo, definir as metas de sucesso do fine-tuning, escolher o modelo base e executar o teste de necessidade.

## 1.1 Definição da tarefa do modelo

### 1.1.1 Política de severidade de triagem de alertas no SOC

A política de severidade da organização depende da combinação do sinal técnico com o contexto:

| Sinal técnico | Ativo crítico (DC, servidor financeiro) | Ativo comum | Ativo de laboratório |
|---|---|---|---|
| Forte | crítica | alta | media |
| Moderado | alta | media | baixa |
| Fraco e recorrente | media | baixa | baixa (falso positivo provável) |

### 1.1.2 Aplicação manual da Política de severidade de triagem de alertas no SOC

Aplicando manualmente temos que para os alertas:

- **A)** EDR: `powershell.exe -enc ...` com processo pai `winword.exe` no host `DC-01` (controlador de domínio). Ocorrências em 7 dias: 1.
- **B)** EDR: execução de `psexec.exe` pelo usuário `m.souza` (analista de RH, sem privilégio admin) na estação `WS-RH-14`. Ocorrências em 7 dias: 1.
- **C)** IDS: varredura de portas a partir do scanner interno `LAB-SCAN-01` no ativo `LAB-WEB-03`. Ocorrências em 7 dias: 120, nenhum incidente associado.
- **D)** EDR: hash com reputação **maliciosa** na Threat Intel executado na máquina `LAB-TEST-07` (laboratório). Ocorrências em 7 dias: 1.

**Solução:**

| Alerta | Sinal técnico | Ativo | Veredito | Severidade | Justificativa |
|---|---|---|---|---|---|
| A | Forte (PowerShell codificado disparado pelo Word) | Crítico | verdadeiro_positivo | **critica** | PowerShell codificado iniciado pelo Word num controlador de domínio é padrão típico de macro maliciosa no ativo mais sensível da rede. |
| B | Moderado (ferramenta administrativa incomum para o perfil) | Comum | verdadeiro_positivo | **media** | PsExec não é esperado para um usuário de RH sem privilégio admin. Pede verificação, mas a estação não é crítica. |
| C | Fraco e recorrente | Laboratório | **falso_positivo** | **baixa** | Varredura recorrente vinda do scanner interno, 120 vezes em 7 dias sem incidente. É atividade autorizada de laboratório. |
| D | Forte (IOC com reputação maliciosa) | Laboratório | verdadeiro_positivo | **media** | O hash é malicioso, mas a máquina é de laboratório e isolada do ambiente produtivo. O sinal forte mantém o veredito positivo. |

### 1.1.3 Definição da tarefa do modelo para aplicação do fine-tuning

Desta forma, define-se a tarefa:

- **Entrada:** Alerta com sinal técnico (fonte, host, usuário, processo) + contexto já enriquecido (criticidade do ativo, perfil do usuário, reputação de IOC, ocorrências em 7 dias)
- **Saída:** JSON com exatamente 3 chaves: `veredito`, `severidade` e `justificativa`

Formato da saída:
```json
{
  "veredito": "falso_positivo | verdadeiro_positivo",
  "severidade": "baixa | media | alta | critica",
  "justificativa": "explicação curta citando o sinal técnico e o contexto"
}
```

### 1.1.4 Seleção do tipo de fine-tuning

Fine-tuning supervisionado (SFT) com LoRA (PEFT) de um LLM pequeno instruction-tuned, com saída em formato restrito.

## 1.2 Definir os critérios de sucesso

Os seguintes critérios de sucesso são:

| Métrica | Meta | Por que essa métrica |
|---|---|---|
| JSON válido com as 3 chaves | ≥ 98% | A triagem é consumida por sistemas (fila do SOC, SOAR). Uma saída que não é JSON válido trava o fluxo. |
| Acurácia de veredito | ≥ 85% | Mede se o modelo separa bem falso positivo de verdadeiro positivo, que é o maior ganho de tempo para o L1. |
| **Recall de severidade "crítica"** | **≥ 90%** | **Métrica de negócio.** De todos os alertas realmente críticos, quantos o modelo marcou como críticos. Um crítico classificado como "media" vai para o fim da fila: é um ataque real ignorado. |
| Justificativa coerente (checklist humano, 10 casos) | ≥ 8 de 10 | O analista só confia na triagem se a justificativa cita o sinal técnico e o contexto e é coerente com a severidade. |
| Ganho sobre o baseline | Superar o baseline | Se o prompt sozinho já resolve, não há por que fazer fine-tuning. |


Portanto temos que:

| Situação | Decisão |
|---|---|
| Baseline atinge **todas** as metas | **Não fazer fine-tuning.** Usar o prompt. |
| Baseline falha em formato (JSON) mas acerta a severidade | O fine-tuning se justifica pela **consistência de formato** |
| Baseline falha em severidade, principalmente em "crítica" | O fine-tuning se justifica por **aprender a política da organização** |
| Baseline falha em ambos | O fine-tuning se justifica pelos dois motivos |


## 1.3 Escolher o modelo base

### 1.3.1 Comparar modelos

Aqui vamos comparar modelos candidatos e escolher um, verificando a licença, tamanho e o acesso no Hub.

| Critério | [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) | [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) | [Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) | [Phi-3.5-mini-instruct](https://huggingface.co/microsoft/Phi-3.5-mini-instruct) |
|---|---|---|---|---|
| Instruction-tuned (chat template) | Sim | Sim | Sim | Sim |
| Licença | Apache 2.0 | Apache 2.0 | Llama 3.2 Community | MIT |
| Acesso *gated* | Não | Não | **Sim** (exige aprovação) | Não |
| Treino em T4 (16GB) | Sim | Sim, mais lento | Sim | Apertado |

Decisão:

- **Escolhido: `Qwen/Qwen2.5-0.5B-Instruct`**, Licença Apache 2.0, sem *gating*, treina rápido numa T4 e é barato de rodar em volume.
- **Alternativa:** `Qwen/Qwen2.5-1.5B-Instruct`, se sobrar tempo ou se a justificativa do 0.5B ficar fraca.
- **Descartados:** Llama 3.2 pelo acesso *gated*, que cria risco de bloqueio no meio do projeto, e Phi-3.5-mini pelo tamanho, que compromete o tempo de treino.

In [1]:
from huggingface_hub import model_info

candidatos = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "meta-llama/Llama-3.2-1B-Instruct",
    "microsoft/Phi-3.5-mini-instruct",
]

for model_id in candidatos:
    info = model_info(model_id)
    licenca = info.card_data.license if info.card_data else None
    params = info.safetensors.total if info.safetensors else None
    print(f"{model_id:40} licença={licenca} gated={info.gated} params={params}")

Qwen/Qwen2.5-0.5B-Instruct               licença=apache-2.0 gated=False params=494032768
Qwen/Qwen2.5-1.5B-Instruct               licença=apache-2.0 gated=False params=1543714304
meta-llama/Llama-3.2-1B-Instruct         licença=llama3.2 gated=manual params=1235814400
microsoft/Phi-3.5-mini-instruct          licença=mit gated=False params=3821079552


### 1.3.2 Carregar o modelo e conferir o chat template

#### 1.3.2.1 Criar conta e Token no Hugging Face

Em [huggingface.co/join](https://huggingface.co/join), criar a conta e confirmar o e-mail. Contas com e-mail não confirmado não conseguem criar tokens.

Em [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens), *Create new token → Write*, dar um nome (ex.: `colab-fine-tuning-soc`) e copiar o valor. O Hub só mostra o token uma vez.

#### 1.3.2.2 Configurar o Google Colab

Abrir o link do notebook e ir em *Arquivo → Salvar uma cópia no Drive*. A partir daqui, trabalhar sempre na cópia.

Em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecionar **Python 3**, **GPU T4**, **Versão 2026.10** e conferir que ela está ativa rodando o código da célula abaixo.

In [2]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Esperado: Tesla T4, 15360 MiB

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


No painel lateral, ícone de chave → *Adicionar novo secret* → nome `HF_TOKEN`, valor = token → ligar *Acesso ao notebook*. Depois, rodar a célula abaixo para verificar a configuração.

In [3]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))

conta = whoami()
print(f"Conta:         {conta['name']}")
print(f"Tipo de token: {conta['auth']['accessToken']['role']}")
# Esperado: o nome da conta e "write"

Conta:         poliana-moraes
Tipo de token: write


#### 1.3.2.3 Instalar as bibliotecas

Instalar as bibliotecas através das linhas de comando abaixo:

In [4]:
# Versões fixadas para o runtime 2026.10 do Colab (ver README)
!pip install -q transformers==5.22.0 datasets==4.8.5 accelerate==1.15.0 huggingface_hub==1.33.0

Verificar que o PyTorch está usando a GPU e quanta memória está livre:

In [5]:
import torch

assert torch.cuda.is_available(), "GPU não encontrada: selecione GPU T4 no tipo de ambiente de execução."

livre, total = torch.cuda.mem_get_info()
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"CUDA (PyTorch): {torch.version.cuda}")
print(f"Memória livre:  {livre / 1e9:.1f} GB de {total / 1e9:.1f} GB")
print(f"Suporta bf16:   {torch.cuda.is_bf16_supported()}")

GPU:            Tesla T4
CUDA (PyTorch): 13.0
Memória livre:  15.5 GB de 15.6 GB
Suporta bf16:   True


#### 1.3.2.4 Teste do ambiente de execução

Executar teste final completo do ambiente:

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

messages = [{"role": "user", "content": "Em uma frase: o que faz um analista de SOC?"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=50, do_sample=False)

print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
print(f"Memória da GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

O que faz um analista de Segurança da Computação (SOC) é um profissional especializado em análise e identificação de riscos relacionados à segurança da informação. Aqui estão alguns pontos-chave sobre o trabalho de um anal
Memória da GPU: 1.00 GB


#### 1.3.2.5 Carregar modelo

Carregar o modelo e verificar o formato de prompt esperado e quanta memória da GPU ele ocupa:

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

exemplo = [
    {"role": "system", "content": "Você é um analista de SOC."},
    {"role": "user", "content": "Alerta de teste"},
]
print(tokenizer.apply_chat_template(exemplo, tokenize=False, add_generation_prompt=True))
print(f"Memória da GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

<|im_start|>system
Você é um analista de SOC.<|im_end|>
<|im_start|>user
Alerta de teste<|im_end|>
<|im_start|>assistant

Memória da GPU: 1.00 GB


#### 1.3.2.6 Definir o prompt few-shot do baseline

Escrever um prompt com a política de severidade e 3 exemplos resolvidos, que será o baseline do projeto. Os exemplos seguem o mesmo formato de alerta e contexto do dataset, com a criticidade escrita como `criticidade: crítico`, `comum` ou `laboratório`:

In [8]:
import json

POLITICA = """Política de severidade da organização:
- Sinal forte: ativo crítico = critica; ativo comum = alta; ativo de laboratório = media.
- Sinal moderado: ativo crítico = alta; ativo comum = media; ativo de laboratório = baixa.
- Sinal fraco e recorrente: ativo crítico = media; ativo comum = baixa; ativo de laboratório = baixa (falso positivo provável).
Responda APENAS com um JSON com as chaves: veredito (falso_positivo | verdadeiro_positivo), severidade (baixa | media | alta | critica) e justificativa."""

SYSTEM_BASELINE = f"Você é um analista de SOC.\n{POLITICA}"

FEW_SHOT = [
    (
        "EDR | Host: DC-01 | Usuário: svc_backup | Processo: powershell.exe -enc JAB... | Pai: winword.exe\n"
        "Ativo: controlador de domínio (criticidade: crítico) | Usuário: conta de serviço | Reputação: desconhecida | Ocorrências em 7 dias: 1",
        {"veredito": "verdadeiro_positivo", "severidade": "critica",
         "justificativa": "PowerShell codificado iniciado pelo Word em controlador de domínio, padrão de macro maliciosa no ativo mais sensível."},
    ),
    (
        "EDR | Host: WS-RH-14 | Usuário: m.souza | Processo: psexec.exe\n"
        "Ativo: estação de trabalho do RH (criticidade: comum) | Usuário: analista de RH, sem privilégio admin | Reputação: ferramenta legítima usada de forma suspeita | Ocorrências em 7 dias: 1",
        {"veredito": "verdadeiro_positivo", "severidade": "media",
         "justificativa": "Ferramenta administrativa incomum para o perfil do usuário, em estação não crítica."},
    ),
    (
        "IDS | Origem: LAB-SCAN-01 | Destino: LAB-WEB-03 | Evento: varredura de portas\n"
        "Ativo: servidor web de homologação (criticidade: laboratório) | Origem: scanner interno autorizado | Ocorrências em 7 dias: 120, nenhum incidente associado",
        {"veredito": "falso_positivo", "severidade": "baixa",
         "justificativa": "Varredura recorrente do scanner interno autorizado em ambiente de laboratório."},
    ),
]


def montar_mensagens(alerta: str, contexto: str) -> list[dict]:
    messages = [{"role": "system", "content": SYSTEM_BASELINE}]
    for entrada, saida in FEW_SHOT:
        messages.append({"role": "user", "content": entrada})
        messages.append({"role": "assistant", "content": json.dumps(saida, ensure_ascii=False)})
    messages.append({"role": "user", "content": f"{alerta}\n{contexto}"})
    return messages

#### 1.3.2.7 Executar o baseline na amostra

Carregar os 20 alertas da amostra de baseline e gerar a triagem de cada um com o prompt criado:

In [9]:
from datasets import load_dataset

DATASET_ID = "poliana-moraes/alertas-soc-triagem"

amostra = load_dataset(DATASET_ID, data_files="hf-dataset/amostra_baseline.jsonl", split="train")
print(len(amostra))  # 20
print(amostra[0])


def gerar(messages: list[dict]) -> str:
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


saidas = [gerar(montar_mensagens(ex["alerta"], ex["contexto"])) for ex in amostra]
print(saidas[0])

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

amostra_baseline.jsonl:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

20
{'template_id': 'amostra_01', 'alerta': 'EDR | Host: SRV-ERP-02 | Usuário: f.lima | Processo: rundll32.exe upd.dll,Start | Pai: excel.exe', 'contexto': 'Ativo: servidor do ERP financeiro (criticidade: crítico) | Usuário: analista contábil, sem privilégio admin | Reputação do hash: maliciosa | Ocorrências em 7 dias: 1', 'triagem': {'veredito': 'verdadeiro_positivo', 'severidade': 'critica', 'justificativa': 'DLL com reputação maliciosa carregada via rundll32 a partir do Excel no servidor do ERP financeiro, ativo crítico.'}}
{"veredito": "verdadeiro_positivo", "severidade": "critica", "justificativa": "Hash malicioso codificado no arquivo UpdaterDLL do Servidor ERP Financeiro, padrão de macro maliciosa na atividade mais crítica."}


#### 1.3.2.8 Medir o resultado do baseline

A avaliação precisa ser **estrita** e igual à que será usada na Fase 6, para que a comparação baseline × fine-tuned seja justa. Saída inválida conta como erro em todas as métricas: na prática, o sistema não consegue usá-la.

Validar cada saída e calcular as métricas da seção 1.2 sobre os 20 casos:

In [10]:
VEREDITOS = {"falso_positivo", "verdadeiro_positivo"}
SEVERIDADES = {"baixa", "media", "alta", "critica"}
CHAVES = {"veredito", "severidade", "justificativa"}


def validar(saida: str) -> dict | None:
    try:
        d = json.loads(saida)
    except json.JSONDecodeError:
        return None
    if not isinstance(d, dict) or set(d) != CHAVES:
        return None
    if d["veredito"] not in VEREDITOS or d["severidade"] not in SEVERIDADES:
        return None
    return d


previstos = [validar(s) for s in saidas]
reais = [ex["triagem"] for ex in amostra]
n = len(reais)

json_valido = sum(p is not None for p in previstos) / n
acc_veredito = sum(p is not None and p["veredito"] == r["veredito"] for p, r in zip(previstos, reais)) / n
acc_severidade = sum(p is not None and p["severidade"] == r["severidade"] for p, r in zip(previstos, reais)) / n

criticos = [(p, r) for p, r in zip(previstos, reais) if r["severidade"] == "critica"]
assert criticos, "a amostra não tem alertas críticos: o recall de crítica não pode ser calculado"
recall_critica = sum(p is not None and p["severidade"] == "critica" for p, r in criticos) / len(criticos)

print(f"JSON válido:         {json_valido:.0%}")
print(f"Acurácia veredito:   {acc_veredito:.0%}")
print(f"Acurácia severidade: {acc_severidade:.0%}")
print(f"Recall de crítica:   {recall_critica:.0%} ({len(criticos)} críticos na amostra)")

JSON válido:         80%
Acurácia veredito:   60%
Acurácia severidade: 50%
Recall de crítica:   80% (5 críticos na amostra)


Também vale inspecionar as falhas, porque elas orientam as próximas fases:

In [11]:
for s, p in zip(saidas, previstos):
    if p is None:
        print("INVÁLIDA:", s[:150])

INVÁLIDA: {"veredito": "verdadeiro_positivo", "severidade": "crítico", "justificativa": "Login RDP realizado às 02:47 a partir do IP interno, indicando atividad
INVÁLIDA: {"veredito": "verdadeiro_positivo", "severidade": "crítico", "justificativa": "Falha de pré-autenticação Kerberos ocorrida por conta própria da autori
INVÁLIDA: {"veredito": "verdadeiro_positivo", "severidade": "crítico", "justificativa": "VSSAdmin.exe deletar cachos de segurança de servidores de backup, padrã
INVÁLIDA: {"veredito": "verdadeiro_positivo", "severidade": "crítico", "justificativa": "AdFind executado por administrador de domínio, que normalmente usa ferr


#### 1.3.2.9 Selecionar o modelo

Comparação do resultado do baseline com as metas:

| Métrica | Meta | Baseline (few-shot, 0.5B) | Atingiu? |
|---|---|---|---|
| JSON válido | ≥ 98% | 80%  | Não |
| Acurácia de veredito | ≥ 85% | 60% | Não |
| Acurácia da severidade | ≥ 90% | 50% | Não |
| Recall de "crítica" | ≥ 90% | 80% | Não |


**Conclusão:** com o melhor prompt few-shot, o modelo 0.5B não atinge nenhuma meta. Ele erra o formato em 20% dos casos e deixa de marcar alertas críticos, porque ignora o contexto do ativo (trata o sinal forte sempre igual). O fine-tuning se justifica pelos dois motivos: consistência de formato e aplicação da política de severidade.